# Valuation-to-Option Pricing Lab

## Student operating instructions
Students update only two input files:

1. `data/company_inputs.csv`: one row containing the firm, valuation, option, and market assumptions.
2. `data/analyst_targets.csv`: one row for each analyst target.

Run all cells. The notebook validates the finance theory and writes `docs/model_results.json`. The existing GitHub Pages workflow then publishes the Plotly dashboard. Students do not edit the workflow or dashboard code.

> Analyst targets feed a subjective valuation distribution. The CRR lattice independently generates the risk-neutral no-arbitrage value.

## 1. Load and validate student inputs

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
assert len(inputs)==1, 'company_inputs.csv must contain exactly one firm row.'
M=inputs.iloc[0].to_dict()
for k in ['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','dcf_value','comps_value']:
    M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower()
assert M['spot']>0 and M['strike']>0 and M['maturity_years']>0 and M['volatility']>0
assert M['market_bid']<=M['market_ask']; assert M['option_type'] in ['call','put']; assert M['exercise_style'] in ['european','american']
targets=pd.read_csv(DATA/'analyst_targets.csv')
assert {'analyst','target','publication_date','horizon_months'}.issubset(targets.columns)
assert len(targets)>=5, 'At least five analyst targets are required.'
assert targets.target.gt(0).all(); assert targets.horizon_months.nunique()==1, 'All analyst targets must use one horizon.'
M

## 2. Equity valuation and target distribution

In [ ]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
pd.Series(valuation)

## 3. Reproducible state construction from analyst quantiles

In [ ]:
# Prices use target quantiles. Weights are disclosed pedagogical scenario masses.
quantiles=targets.target.quantile([.10,.50,.80,.95]).to_numpy(float)
states=pd.DataFrame({'state':['Bear','Base','Bull','Tail'],'terminal_price':quantiles,'subjective_weight':[.20,.45,.25,.10]})
assert np.isclose(states.subjective_weight.sum(),1); assert states.subjective_weight.ge(0).all()
states

## 4. Contingent payoffs and subjective expected payoff

In [ ]:
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float)
    return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type'])
states['weighted_payoff']=states.subjective_weight*states.option_payoff
belief_value=float(np.exp(-M['risk_free_rate']*M['maturity_years'])*states.weighted_payoff.sum())
states

## 5. CRR risk-neutral lattice

In [ ]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt)
    p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
lattice

## 6. Theoretical validation

In [ ]:
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk))
steps=[25,50,100,200,500]; conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2])}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
validation

## 7. Export dashboard data

In [ ]:
results={'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(v)} for n,v in zip(steps,conv)],'comparison':{'Valuation-weighted expected payoff':belief_value,'Risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']}}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2))
print('Dashboard data exported:',DOCS/'model_results.json')
results['comparison']